# Notebook 2 · Feature Selection for Linear Models

Learn how to choose useful input columns using correlation, plots, transforms, adjusted R² and standardised coefficients, with pandas and scikit-learn.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/notebooks/nb-02-feature-selection.ipynb)

- Lecture: [en-02-feature-selection.md](https://github.com/Rathachai/DA-LAB/blob/gh-pages/learnings/linear/en-02-feature-selection.md)
- Interactive lab(s): [CORR101](https://rathachai.github.io/DA-LAB/learnings/linear/corr101.html) · [LM201](https://rathachai.github.io/DA-LAB/learnings/linear/lm201.html)

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score
pd.options.display.float_format = '{:.3f}'.format
print('Libraries ready')

## 2. Correlation patterns: `corr101_patterns.csv`
One file with nine different shapes of data, 100 points each.

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/corr101_patterns.csv"
pat = pd.read_csv(url)
pat.head()

How many points does each pattern have?

In [ ]:
pat['pattern'].value_counts().to_frame('points')

One scatter plot per pattern, with the Pearson correlation `r` in each title. `r` runs from -1 to +1 and measures **straight-line** association only.

In [ ]:
names = list(pat['pattern'].unique())
fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for ax, n in zip(axes.ravel(), names):
    g = pat[pat['pattern'] == n]
    ax.scatter(g['x'], g['y'], s=10)
    ax.set_title(f"{n}  (r = {g['x'].corr(g['y']):.2f})", fontsize=10)
    ax.set_xlabel('x'); ax.set_ylabel('y')
plt.tight_layout(); plt.show()

The same numbers as a table. Notice: **circle, U-shape and sine wave have r close to 0** even though the pattern is obvious; **one outlier** can change r a lot; **two clusters** can look strongly correlated although there is no trend inside each cluster. Always plot your data.

In [ ]:
r_tab = pd.Series({n: g['x'].corr(g['y']) for n, g in pat.groupby('pattern')})
r_tab.to_frame('Pearson r').sort_values('Pearson r')

## 3. Correlation by hand
For five points: `r = Sxy / sqrt(Sxx * Syy)`. Then check with `np.corrcoef`.

In [ ]:
small = pd.DataFrame({'x': [10, 20, 30, 40, 50], 'y': [52, 61, 68, 80, 88]})
dx, dy = small['x'] - small['x'].mean(), small['y'] - small['y'].mean()
r_hand = (dx * dy).sum() / np.sqrt((dx**2).sum() * (dy**2).sum())
r_np = np.corrcoef(small['x'], small['y'])[0, 1]
print('r by hand  :', round(r_hand, 5))
print('r np.corrcoef:', round(r_np, 5))
small.assign(dx=dx, dy=dy, dx_dy=dx * dy)

Two handy links: the **slope** of the fitted line is `r * sy / sx`, and **R² = r²** for one feature. We confirm both with `LinearRegression`.

In [ ]:
lr1 = LinearRegression().fit(small[['x']], small['y'])
pd.DataFrame({'from r': [r_np * small['y'].std() / small['x'].std(), r_np**2],
              'from LinearRegression': [lr1.coef_[0], lr1.score(small[['x']], small['y'])]},
             index=['slope (°C/kW)', 'R²'])

## 4. A realistic data set: `lm201_data.csv`
Columns `id`, `x1`...`x7` (candidate features) and `y` (target). Which x columns help to predict y?

In [ ]:
url = "https://rathachai.github.io/DA-LAB/learnings/linear/data/lm201_data.csv"
df = pd.read_csv(url)
df.head()

Summary statistics.

In [ ]:
df.describe()

Correlation of every column with `y` (we include `id` on purpose: it should be useless).

In [ ]:
r_y = df.drop(columns='y').corrwith(df['y']).rename('r with y')
display(r_y.to_frame())
colors = ['tab:green' if v > 0 else 'tab:red' for v in r_y]
plt.bar(r_y.index, r_y.values, color=colors)
plt.axhline(0, color='black', lw=0.8)
plt.title('Pearson r of each column with y'); plt.xlabel('Column'); plt.ylabel('r'); plt.show()

Plot every x against y. Look for straight lines (x1, x2, x6, x7), a U-shape (x3), pure noise (x4) and a curve (x5). `id` is added as well.

In [ ]:
cols = ['id', 'x1', 'x2', 'x3', 'x4', 'x5', 'x6', 'x7']
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.ravel(), cols):
    ax.scatter(df[col], df['y'], s=10)
    ax.set_title(f"{col}  (r = {df[col].corr(df['y']):.2f})", fontsize=10)
    ax.set_xlabel(col); ax.set_ylabel('y')
plt.tight_layout(); plt.show()

## 5. The log transform for x5
`x5` grows exponentially, so its logarithm should be straighter. We add a column `ln_x5` (log needs positive values) and compare.

In [ ]:
print('min x5 =', df['x5'].min())
df['ln_x5'] = np.log(df['x5'])
df[['id', 'x5', 'ln_x5', 'y']].head()

Before and after: compare `r` and the scatter plots.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
for a, col in zip(ax, ['x5', 'ln_x5']):
    a.scatter(df[col], df['y'], s=10)
    a.set_title(f"{col}: r = {df[col].corr(df['y']):.2f}")
    a.set_xlabel(col); a.set_ylabel('y')
plt.tight_layout(); plt.show()

## 6. Multiple regression with several feature sets
We fit `LinearRegression` on different column sets. **R²** never decreases when you add a column (even a noise column), but **adjusted R²** = `1 - (1-R²)(n-1)/(n-p-1)` penalises useless features (`n` = rows, `p` = number of features). The U-shaped `x3` is added raw, and `ln_x5` is the transformed x5. (Here the score is on the same data used to fit, to isolate the effect of adding columns.)

In [ ]:
sets = {'x1 only': ['x1'],
        'x1 + x7': ['x1', 'x7'],
        'good: x1 x2 x6 x7': ['x1', 'x2', 'x6', 'x7'],
        'good + x4 (noise)': ['x1', 'x2', 'x6', 'x7', 'x4'],
        'good + x3 (U-shape)': ['x1', 'x2', 'x6', 'x7', 'x3'],
        'good + ln(x5)': ['x1', 'x2', 'x6', 'x7', 'ln_x5']}
rows, n = [], len(df)
for name, f in sets.items():
    model = LinearRegression().fit(df[f], df['y'])
    r2 = r2_score(df['y'], model.predict(df[f]))
    adj = 1 - (1 - r2) * (n - 1) / (n - len(f) - 1)
    rows.append({'feature set': name, 'p': len(f), 'R2': r2, 'adjusted R2': adj})
res = pd.DataFrame(rows).set_index('feature set')
res

Bar chart of the two scores. Adding noise `x4` or U-shaped `x3` barely moves R² and does not help adjusted R². Adding `ln(x5)` is the one that really helps.

In [ ]:
res[['R2', 'adjusted R2']].plot.bar(figsize=(9, 4), ylim=(res['R2'].min() - 0.05, 1))
plt.title('R² and adjusted R² for each feature set')
plt.ylabel('Score'); plt.xlabel('Feature set'); plt.xticks(rotation=30, ha='right')
plt.tight_layout(); plt.show()

## 7. Coefficients: original units vs standardised
In original units a coefficient is 'change in y per one unit of that x', so columns with different units are **not comparable** (x2 is large in number, x1 small). After `StandardScaler` every feature has mean 0 and std 1, so each coefficient means 'change in y per one standard deviation', which can be compared.

In [ ]:
feats = ['x1', 'x2', 'x6', 'x7', 'ln_x5']
raw = LinearRegression().fit(df[feats], df['y'])
pipe = make_pipeline(StandardScaler(), LinearRegression()).fit(df[feats], df['y'])
coefs = pd.DataFrame({'std of feature': df[feats].std(),
                      'coef (original units)': raw.coef_,
                      'coef (standardised)': pipe[-1].coef_}, index=feats)
coefs

Plot the standardised coefficients: the longest bar is the most influential feature.

In [ ]:
coefs['coef (standardised)'].sort_values().plot.barh(color='tab:blue')
plt.axvline(0, color='black', lw=0.8)
plt.title('Standardised coefficients (change in y per 1 std of x)')
plt.xlabel('Coefficient (y units per std)'); plt.ylabel('Feature'); plt.show()
print('Same predictions either way:', np.allclose(raw.predict(df[feats]), pipe.predict(df[feats])))

## 8. Leakage demo
**Data leakage** means giving the model information it would not have in real use. Adding `y` itself (or something computed from it) makes R² jump to 1 and teaches nothing. Adding `id` is meaningless: its weight is just an accident.

In [ ]:
base = ['x1', 'x2', 'x6', 'x7', 'ln_x5']
df['y_copy'] = df['y']                       # the target disguised as a feature
tests = {'honest features': base, '+ id': base + ['id'], '+ y (leak!)': base + ['y_copy']}
rows = []
for name, f in tests.items():
    mdl = LinearRegression().fit(df[f], df['y'])
    rows.append({'feature set': name, 'R2': mdl.score(df[f], df['y']),
                 'weight of last column': mdl.coef_[-1]})
pd.DataFrame(rows).set_index('feature set')

Show the leaked model in detail: the weight on the copy of `y` is 1 and everything else collapses to 0.

In [ ]:
f = base + ['y_copy']
leak = LinearRegression().fit(df[f], df['y'])
pd.Series(leak.coef_, index=f, name='coefficient').to_frame()

## 9. Summary / Key takeaways

- Pearson r only measures straight-line association: circle, U-shape and sine give r near 0; an outlier or two clusters can fool r. Always plot.
- For one feature: slope = r * sy / sx and R² = r².
- Rank columns by r, but check scatter plots: x3 (U-shape) is a trap, x4 is noise, id is a label.
- A log transform straightens an exponential feature (x5), raising |r| and R².
- R² never decreases when you add features; adjusted R² can, so use it to compare feature sets.
- Standardised coefficients are comparable across features; original-unit coefficients are not.
- Never use y or id as a feature (leakage).

Powered by: Sonnet  |  AI Whisperer: rathachai.github.io